# Exploratory Data Analysis — RACE Dataset
**Course:** AL2002 — Artificial Intelligence | **Institution:** FAST-NUCES Islamabad

This notebook performs EDA on the RACE Reading Comprehension dataset, analyzing passage lengths, answer distributions, and question types.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import os

sns.set_theme(style='darkgrid')
print('Libraries loaded.')

In [ ]:
# Load the raw dataset
train_df = pd.read_csv('../data/raw/train.csv')

print(f'Total rows: {len(train_df)}')
print(f'Columns   : {list(train_df.columns)}')
print('\nFirst row:')
train_df.head(1)

In [ ]:
# Dataset Shape and Missing Values
cols = ['id', 'article', 'question', 'A', 'B', 'C', 'D', 'answer']
print('=== Dataset Summary ===')
print(f'Total Samples   : {len(train_df):,}')
print(f'Total Columns   : {len(train_df.columns)}')
print('\nMissing Values per Column:')
print(train_df[cols].isnull().sum())

In [ ]:
# --- CHART 1: Answer Label Distribution ---
fig, ax = plt.subplots(figsize=(7, 4))
counts = train_df['answer'].value_counts().reindex(['A', 'B', 'C', 'D'])
bars = ax.bar(['A', 'B', 'C', 'D'], counts.values,
               color=['#4c8bf5', '#34a853', '#fbbc05', '#ea4335'], edgecolor='white', width=0.6)

for bar, count in zip(bars, counts.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 100,
            f'{count:,}\n({count/len(train_df)*100:.1f}%)',
            ha='center', va='bottom', fontsize=11, fontweight='bold')

ax.set_title('Distribution of Correct Answers (A / B / C / D)', fontsize=14, fontweight='bold')
ax.set_xlabel('Answer Option', fontsize=12)
ax.set_ylabel('Number of Questions', fontsize=12)
ax.set_ylim(0, counts.max() * 1.18)
plt.tight_layout()
os.makedirs('../report', exist_ok=True)
plt.savefig('../report/answer_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: report/answer_distribution.png')

In [ ]:
# --- CHART 2: Passage Length Distribution ---
article_lengths = train_df['article'].apply(lambda x: len(str(x).split()))
question_lengths = train_df['question'].apply(lambda x: len(str(x).split()))

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].hist(article_lengths, bins=60, color='#4c8bf5', edgecolor='white', alpha=0.85)
axes[0].axvline(article_lengths.mean(), color='red', linestyle='--',
                label=f'Mean: {article_lengths.mean():.0f} words')
axes[0].set_title('Passage Length Distribution (Word Count)', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Number of Words')
axes[0].set_ylabel('Frequency')
axes[0].legend()

axes[1].hist(question_lengths, bins=40, color='#34a853', edgecolor='white', alpha=0.85)
axes[1].axvline(question_lengths.mean(), color='red', linestyle='--',
                label=f'Mean: {question_lengths.mean():.0f} words')
axes[1].set_title('Question Length Distribution (Word Count)', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Number of Words')
axes[1].set_ylabel('Frequency')
axes[1].legend()

plt.tight_layout()
plt.savefig('../report/length_distributions.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'Avg passage length : {article_lengths.mean():.0f} words')
print(f'Max passage length : {article_lengths.max()} words')
print(f'Avg question length: {question_lengths.mean():.0f} words')
print('Saved: report/length_distributions.png')

In [ ]:
# --- CHART 3: Question Type Analysis (Wh-word distribution) ---
wh_words = ['what', 'why', 'how', 'who', 'where', 'when', 'which']
wh_counts = {}
for wh in wh_words:
    count = train_df['question'].str.lower().str.startswith(wh).sum()
    wh_counts[wh.capitalize()] = count

other = len(train_df) - sum(wh_counts.values())
wh_counts['Other'] = other

fig, ax = plt.subplots(figsize=(8, 4))
colors = ['#4c8bf5','#34a853','#fbbc05','#ea4335','#9b59b6','#e67e22','#1abc9c','#95a5a6']
bars = ax.bar(wh_counts.keys(), wh_counts.values(), color=colors, edgecolor='white')

for bar, val in zip(bars, wh_counts.values()):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 100,
            f'{val:,}', ha='center', fontsize=9, fontweight='bold')

ax.set_title('Question Type Distribution (First Word)', fontsize=13, fontweight='bold')
ax.set_xlabel('Question Word')
ax.set_ylabel('Count')
plt.tight_layout()
plt.savefig('../report/question_types.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: report/question_types.png')

In [ ]:
# --- SUMMARY STATISTICS TABLE ---
summary = pd.DataFrame({
    'Metric': [
        'Total Questions', 'Train Split (80%)', 'Val Split (10%)', 'Test Split (10%)',
        'Avg Passage Length (words)', 'Max Passage Length (words)',
        'Avg Question Length (words)', 'Answer A (%)', 'Answer B (%)',
        'Answer C (%)', 'Answer D (%)'
    ],
    'Value': [
        f"{len(train_df):,}", '70,292', '8,787', '8,787',
        f"{article_lengths.mean():.0f}", f"{article_lengths.max()}",
        f"{question_lengths.mean():.0f}",
        f"{(train_df['answer']=='A').mean()*100:.1f}%",
        f"{(train_df['answer']=='B').mean()*100:.1f}%",
        f"{(train_df['answer']=='C').mean()*100:.1f}%",
        f"{(train_df['answer']=='D').mean()*100:.1f}%"
    ]
})

print('=== Dataset Summary Statistics ===')
print(summary.to_string(index=False))